In [2]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, BertModel, BertTokenizerFast
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# if using Google Colab, mount Google Drive
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    train_path = '/content/drive/MyDrive/CS5242Project/train_preprocessed.csv'
    test_path  = '/content/drive/MyDrive/CS5242Project/test_preprocessed.csv'
else:
    train_path = 'transformers_train_preprocessed.csv'
    test_path  = 'transformers_test_preprocessed.csv'

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

train['cleaned_text'] = train['cleaned_text'].astype(str)
test['cleaned_text']  = test['cleaned_text'].astype(str)

from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train['Bias'])
y_test  = label_encoder.transform(test['Bias'])
num_classes = len(label_encoder.classes_)

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encodings = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encodings["input_ids"].squeeze(0),
            "attention_mask": encodings["attention_mask"].squeeze(0),
            "labels": torch.tensor(label)
        }

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
train_dataset = NewsDataset(train['cleaned_text'], y_train, tokenizer)
test_dataset = NewsDataset(test['cleaned_text'], y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

In [4]:
from transformers import BertModel, BertForSequenceClassification
from torch import nn

class Bert(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, return_dict=True)
        pooled = outputs.pooler_output
        logits = self.classifier(pooled)
        return logits

In [5]:
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

model = Bert(num_labels=num_classes)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 3
total_steps = len(train_loader) * num_epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# 6. Training & evaluation loop
for epoch in range(num_epochs):
    # — Training
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        # Backward pass
        loss.backward()
        optimizer.step()
        scheduler.step()

        optimizer.zero_grad()

        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Epoch {epoch+1} →  training loss: {avg_train_loss:.4f}")

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids     = batch["input_ids"].to(device)
            attention_mask= batch["attention_mask"].to(device)
            labels        = batch["labels"].to(device)

            logits = model(input_ids, attention_mask=attention_mask)
            preds  = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    print(f"Epoch {epoch+1} — Eval accuracy: {acc:.4f}; F1: {f1:.4f}; Precision: {precision:.4f}; Recall: {recall:.4f}")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1 →  training loss: 1.5953
Epoch 1 — Eval accuracy: 0.4045; F1: 0.4034; Precision: 0.4871; Recall: 0.4045
Epoch 2 →  training loss: 1.3222
Epoch 2 — Eval accuracy: 0.5236; F1: 0.5538; Precision: 0.6252; Recall: 0.5236
Epoch 3 →  training loss: 0.9952
Epoch 3 — Eval accuracy: 0.5865; F1: 0.6048; Precision: 0.6413; Recall: 0.5865


In [6]:
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))

              precision    recall  f1-score   support

      center       0.45      0.60      0.52        40
   lean left       0.40      0.46      0.43        61
  lean right       0.30      0.41      0.34        37
        left       0.87      0.67      0.75       230
       right       0.43      0.52      0.47        77

    accuracy                           0.59       445
   macro avg       0.49      0.53      0.50       445
weighted avg       0.64      0.59      0.60       445

